# Imports

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)   # generator naključnih števil (ponovljivost)

# Parametri energije
K, Gama, lam = 1.0, 1.0, 1.0     # utež ploščine, obsega in dolžin stranic
R = 1.0                          # polmer očrtanega kroga

# Funkcije

In [ ]:
def poligon(R, n, eps=0.0, rng=rng):
    """Oglišča n-kotnika, vrisanega v krog s polmerom R (v pozitivni smeri).
    eps: relativni šum kota in polmera. Vrne array (n, 2)."""
    dfi = 2 * np.pi / n
    fi = dfi * np.arange(n) + rng.uniform(-eps, eps, n) * dfi
    r = R * (1 + rng.uniform(-eps, eps, n))
    return np.column_stack([r * np.cos(fi), r * np.sin(fi)])


def stranice(t):
    """Dolžine stranic poligona t (zadnje oglišče povežemo s prvim)."""
    # TODO: np.roll
    pass


def obseg(t):
    # TODO
    pass


def ploscina(t):
    """Ploščina po formuli vezalk (shoelace)."""
    # TODO
    pass


def tezisce(t):
    """Masno središče oglišč."""
    # TODO
    pass


def referencne(R, n):
    """Analitične vrednosti za pravilni n-kotnik: (A0, P0, l0)."""
    # TODO (P1)
    pass


def energija(t, ref, K=K, Gama=Gama, lam=lam):
    """Mehanska energija celice; ref = (A0, P0, l0)."""
    A0, P0, l0 = ref
    # TODO
    pass


def relaksacija_mc(t, ref, n_iter=20_000, korak=0.05, T=0.0, prilagodi=True, p_cilj=(0.2, 0.3), rng=rng):
    """Relaksacija z metodo Monte Carlo. T = 0: pohlepni algoritem, T > 0: Metropolis.
    p_cilj: interval želenega deleža sprejetih potez (prilagajanje koraka).
    Vrne (t, E, delez_sprejetih, korak) kot arraye po iteracijah."""
    t = t.copy()
    E = energija(t, ref)
    zgod_E, zgod_p, zgod_k = [], [], []
    p = 0.5                       # drseče povprečje deleža sprejetih potez
    for it in range(1, n_iter + 1):
        i = rng.integers(len(t))
        staro = t[i].copy()
        t[i] += rng.uniform(-korak, korak, 2)
        E_nova = energija(t, ref)
        dE = E_nova - E
        # TODO: kriterij sprejetja (pohlepni za T = 0, Metropolis za T > 0)
        sprejmi = False
        if sprejmi:
            E = E_nova
        else:
            t[i] = staro
        p = 0.99 * p + 0.01 * sprejmi
        if prilagodi:   # delež sprejetih držimo v intervalu p_cilj
            # TODO: zmanjšaj korak (* 0.99), če je p < p_cilj[0], povečaj (* 1.01), če je p > p_cilj[1]
            pass
        zgod_E.append(E)
        zgod_p.append(p)
        zgod_k.append(korak)
    return t, np.array(zgod_E), np.array(zgod_p), np.array(zgod_k)


def narisi_celico(ax, t, barva="tab:orange", **kw):
    """Nariše poligon in njegovo težišče."""
    z = np.vstack([t, t[:1]])
    ax.fill(z[:, 0], z[:, 1], color=barva, alpha=0.3)
    ax.plot(z[:, 0], z[:, 1], "o-", color=barva, **kw)
    ax.plot(*tezisce(t), "k+", ms=10)
    ax.set_aspect("equal")

# Testno območje

In [ ]:
for n in (4, 6):
    t = poligon(R, n)
    print(f"n = {n}: A = {ploscina(t):.4f}, P = {obseg(t):.4f}, l = {stranice(t)[0]:.4f}, "
          f"ref = {np.round(referencne(R, n), 4)}, E = {energija(t, referencne(R, n)):.2e}")

# P3: pravokotnik
t_p = np.array([[0, 0], [2, 0], [2, 1], [0, 1]], dtype=float)
print("pravokotnik: A =", ploscina(t_p), " težišče =", tezisce(t_p))

# P4: šestkotnik, povečan za faktor 1.1
print("E(1.1 * šestkotnik) =", energija(1.1 * poligon(R, 6), referencne(R, 6)))

# Naloga 1 - Slika 2

In [ ]:
fig = plt.figure(figsize=(12, 3))
for k, n in enumerate(range(3, 9)):
    ax = fig.add_subplot(1, 6, k + 1)
    # TODO: pravilni n-kotnik in naključno popačen (eps = 0.2)
    ax.set_title(f"n = {n}")
fig.tight_layout()
plt.show()

# Naloga 1 - Slika 3

In [ ]:
n_v = np.arange(3, 101)
# TODO: razmerji A(n)/(pi R^2) in P(n)/(2 pi R) iz numerično izračunanih poligonov
razmerje_A = razmerje_P = None
fig = plt.figure(figsize=(7, 4))
ax = fig.add_subplot(111)
# TODO: obe razmerji v odvisnosti od n
plt.show()
# Preglednica 1
for n in (3, 4, 6, 12, 100):
    print(n, np.round(referencne(R, n)[:2], 4), round(ploscina(poligon(R, n)) / np.pi, 4), round(obseg(poligon(R, n)) / (2 * np.pi), 4))

# Naloga 2 - Slika 4

In [ ]:
ref6 = referencne(R, 6)
eps_v = np.linspace(0, 0.5, 11)
# TODO: povprečna energija 200 naključnih šestkotnikov za vsak eps
E_eps = None
fig = plt.figure(figsize=(7, 4))
ax = fig.add_subplot(111)
# TODO: <E> v odvisnosti od eps (log-log) in naklon
plt.show()

In [ ]:
# Preglednica 2: prispevki k energiji za en popačen šestkotnik (eps = 0.3)
t0 = poligon(R, 6, eps=0.3, rng=np.random.default_rng(7))
A0, P0, l0 = ref6
# TODO

# Naloga 3

In [ ]:
t_kon, E_h, p_h, k_h = relaksacija_mc(t0, ref6, n_iter=20_000, korak=0.05)
print(f"E: {energija(t0, ref6):.4f} -> {E_h[-1]:.2e}")

# Naloga 3 - Slika 5

In [ ]:
fig = plt.figure(figsize=(15, 3.5))
ax = fig.add_subplot(141)
# TODO: začetna in relaksirana celica; E, delež sprejetih in korak v odvisnosti od iteracije
fig.tight_layout()
plt.show()

In [ ]:
# Preglednica 3: relaksacija za različne n
# TODO

# Naloga 4 - Slika 6

In [ ]:
temperature = [0, 1e-4, 1e-3, 1e-2]
fig = plt.figure(figsize=(7, 4))
ax = fig.add_subplot(111)
# TODO: Metropolis pri različnih T (brez prilagajanja koraka, korak = 0.02); Preglednica 4
plt.show()